In [ ]:
# =========================================================================
# Wan2.1-I2V-14B-480P 圖生影換裝視頻生成 Worker (首幀合格母圖對齊版)
# =========================================================================
import os, sys, io, time, gc, json, base64
import torch, psutil
from PIL import Image

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
gpu_name = f"{torch.cuda.get_device_name(0)} (x{gpu_count})" if gpu_count > 0 else "CPU"
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2) if gpu_count > 0 else 0
ram_gb = round(psutil.virtual_memory().total / (1024**3), 2)
print(f"✅ 硬體檢測完成 | GPU: {gpu_name} | 單卡 VRAM: {vram_gb} GB | System RAM: {ram_gb} GB")

print("📦 正在安裝與確認 Wan2.1 視頻生成專用依賴庫...")
import subprocess
subprocess.run(["pip", "install", "-q", "diffusers>=0.41.0", "transformers>=5.17.0", "accelerate", "imageio[ffmpeg]"], check=False)

from diffusers import WanImageToVideoPipeline
from diffusers.utils import export_to_video

model_id = "Wan-AI/Wan2.1-I2V-14B-480P-Diffusers"
print(f"🎬 [新世代開源頂級視頻模型] 正在載入萬相 Wan2.1: {model_id}...")

# 啟用 CPU offload 確保 16GB VRAM 零負擔流暢運行
pipe = WanImageToVideoPipeline.from_pretrained(
    model_id, torch_dtype=torch.bfloat16
)
pipe.enable_model_cpu_offload()

# 載入內嵌之通過 7 項品質檢驗的首幀 2K 母圖
print("🖼️ 正在解碼首幀母圖 (Base64 零依賴安全讀取)...")
b64_data = "..."
input_img = Image.open(io.BytesIO(base64.b64decode(b64_data))).convert("RGB")
orig_w, orig_h = input_img.size
print(f"   首幀原始解析度: {orig_w}x{orig_h}")

# 鎖定 Wan2.1 直式最佳畫質規格 (480x624，寬高皆為 16 倍數，精準契合 0.778 縱橫比，絕不拉伸)
vid_w, vid_h = 480, 624
input_img = input_img.resize((vid_w, vid_h), Image.LANCZOS)
print(f"   縮放至視頻標準解析度: {vid_w}x{vid_h}")

motion_prompt = "Three young Eurasian women wearing sheer tulle mesh and floral lace lingerie sets in front of a large mirror, gently pulling up and fitting delicate lace straps over their shoulders as they finish dressing, brushing long wavy hair back with their fingers, turning their torsos naturally to admire their hourglass curves in the mirror, realistic body movement and soft eye blinks."
negative_prompt = "nude, nudity, naked, topless, exposed nipples, areola, static image, jerky motion, distorted face, blurry, low quality, morphing limbs"

print("⏳ 正在啟動 Wan2.1 圖生影管線 (81 幀 / ~5 秒 16fps 高階合規換裝動態)...")
t0 = time.time()
video_frames = pipe(
    image=input_img,
    prompt=motion_prompt,
    negative_prompt=negative_prompt,
    height=vid_h,
    width=vid_w,
    num_frames=81,
    guidance_scale=5.0,
    num_inference_steps=30
).frames[0]

dt = round(time.time() - t0, 2)
print(f"✨ 視頻生成完畢！耗時: {dt} 秒，正在輸出 MP4...")

out_vid_name = "changing_clothes_wan21.mp4"
export_to_video(video_frames, out_vid_name, fps=16)
vid_sz_mb = round(os.path.getsize(out_vid_name) / (1024 * 1024), 2)
print(f"🎉 影片已成功輸出為 {out_vid_name} (體積: {vid_sz_mb} MB)！")

bench = {
    "engine": "Wan 2.1-I2V-14B-480P-Diffusers",
    "gpu": gpu_name,
    "elapsed_sec": dt,
    "frames": 81,
    "fps": 16,
    "resolution": f"{vid_w}x{vid_h}",
    "video_file": out_vid_name,
    "file_size_mb": vid_sz_mb
}
with open("video_benchmark_report.json", "w", encoding="utf-8") as bf:
    json.dump(bench, bf, indent=2)
